# Results cleaning

Cleans the **results** table for 2021 to 2025: one row per driver per race.

**Input:** `combined/results_2021.csv` ... `results_2025.csv` (and optionally `laps_<year>.csv` for a cross-check)
**Output:** `processed/results_clean.csv`

**Key columns (the team's join keys):** `year`, `round`, `session`, `driver` (the 3-letter code, same as `Driver` in the laps table).

**What the cleaning does**
1. Stacks the five years, drops unused columns (headshot URL, colours, empty qualifying columns) and removes duplicate rows.
2. Pit-lane starters: FastF1 gives them `GridPosition = 0`. They are flagged, and given the back-of-grid slot.
3. Classification: separates `R` (retired), `W` (withdrawn), `D` (disqualified) from real finishing positions.
4. Groups the free-text `Status` into Finished, Lapped, Mechanical, Accident/Collision, Disqualified, Withdrawn/DNS, Retired (unspecified), Other.
5. Builds the retirement flags used by the event models: `finished_race`, `retired`, `did_not_start`, `disqualified`, plus laps completed.
6. Splits `Time_s` into the winner's race time and every other driver's gap to the winner.
7. Flags reduced-points races (for example Belgium 2021, which scored half points).
8. Adds `positions_gained`, `win`, `podium`, `top10`, and a team lineage column (`team_group`) so renamed teams stay together across years.

Run the cells from top to bottom.

## 1. Settings

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

YEARS = [2021, 2022, 2023, 2024, 2025]
COMBINED_DIR = Path("combined")     # where results_2021.csv etc. are
OUT_DIR = Path("processed")         # where results_clean.csv will be written
OUT_DIR.mkdir(exist_ok=True)

KEY3 = ["year", "round", "session"]     # identifies one race
KEY4 = KEY3 + ["driver"]                # identifies one driver in one race

## 2. Load the five years

In [2]:
frames = []
for y in YEARS:
    p = COMBINED_DIR / f"results_{y}.csv"
    if p.exists():
        d = pd.read_csv(p, low_memory=False)
        print(f"{y}: {len(d)} rows, {d['round'].nunique()} races")
        frames.append(d)
    else:
        print(f"{y}: FILE MISSING ({p})")

raw = pd.concat(frames, ignore_index=True)
print("\nTotal:", raw.shape)
raw.head()

2021: FILE MISSING (combined\results_2021.csv)
2022: FILE MISSING (combined\results_2022.csv)
2023: FILE MISSING (combined\results_2023.csv)
2024: FILE MISSING (combined\results_2024.csv)
2025: FILE MISSING (combined\results_2025.csv)


ValueError: No objects to concatenate

## 3. Look at the raw data first

In [ ]:
print("Columns with missing values:")
print(raw.isna().sum()[lambda s: s > 0].to_string())

In [ ]:
print("Status values:")
print(raw["Status"].value_counts().to_string())

In [ ]:
print("ClassifiedPosition values that are not a number:")
print(raw["ClassifiedPosition"][~raw["ClassifiedPosition"].astype(str).str.fullmatch(r"\d+")].value_counts().to_string())
print("\nRows with GridPosition == 0 (pit-lane starts):", (raw["GridPosition"] == 0).sum())

## 4. Keep the columns we need and standardise names

In [ ]:
KEEP = {
    "year": "year", "round": "round", "session": "session",
    "event": "event", "country": "country", "location": "location", "event_date": "event_date",
    "Abbreviation": "driver", "DriverId": "driver_id", "DriverNumber": "driver_number", "FullName": "full_name",
    "TeamName": "team", "TeamId": "team_id",
    "GridPosition": "grid_position_raw", "Position": "finish_position",
    "ClassifiedPosition": "classification", "Status": "status_raw",
    "Points": "points", "Laps": "laps_completed", "Time_s": "time_s",
}
for c in KEEP:                      # a column missing in some year must not break the notebook
    if c not in raw.columns:
        raw[c] = np.nan

df = raw[list(KEEP)].rename(columns=KEEP).copy()
df["driver"] = df["driver"].astype("string").str.strip().str.upper()
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

before = len(df)
df = df.drop_duplicates(KEY4)
print(f"Removed {before - len(df)} duplicate rows. Rows now: {len(df)}")
print("Rows without a driver code:", df["driver"].isna().sum())
df = df.dropna(subset=["driver"])

## 5. Team lineage
Teams were renamed during 2021 to 2025 (Alfa Romeo to Kick Sauber, AlphaTauri to RB to Racing Bulls). `team_group` keeps each team under one name so the team effect in the models stays continuous.

In [ ]:
def team_group(name):
    if pd.isna(name):
        return np.nan
    t = str(name).lower()
    if "alfa" in t or "sauber" in t:
        return "Sauber"
    if "tauri" in t or "racing bulls" in t or t in ("rb", "rb f1 team") or "visa" in t:
        return "RB"
    if "red bull" in t:
        return "Red Bull"
    if "haas" in t:
        return "Haas"
    if "aston" in t:
        return "Aston Martin"
    return str(name)

df["team_group"] = df["team"].map(team_group)
df.groupby(["year", "team_group"]).size().unstack(fill_value=0)

## 6. Grid position
FastF1 gives pit-lane starters `GridPosition = 0`. They are flagged, and for the grid-to-finish analysis they get the slot behind the last gridded car.

In [ ]:
df["grid_position_raw"] = pd.to_numeric(df["grid_position_raw"], errors="coerce")
df["pit_lane_start"] = df["grid_position_raw"].eq(0)

last_grid = df.groupby(KEY3)["grid_position_raw"].transform("max")
df["grid_position"] = np.where(df["pit_lane_start"], last_grid + 1, df["grid_position_raw"])
df["grid_missing"] = df["grid_position"].isna()

print("Pit-lane starts per year:")
print(df.groupby("year")["pit_lane_start"].sum().to_string())
print("\nRows with no grid position:")
df[df["grid_missing"]][["year", "round", "driver", "classification", "status_raw", "laps_completed"]]

## 7. Classification and finishing position
`ClassifiedPosition` is a number for classified drivers, or a letter code: `R` retired, `W` withdrawn, `D` disqualified, `N` not classified, `F` failed to qualify.

In [ ]:
df["finish_position"] = pd.to_numeric(df["finish_position"], errors="coerce")
df["classified_position"] = pd.to_numeric(df["classification"], errors="coerce")
df["classification_code"] = np.where(df["classified_position"].notna(), "CLASSIFIED",
                                     df["classification"].astype("string").str.upper())
df["classification_code"].value_counts()

## 8. Group the free-text `Status`
Edit the keyword lists if the table at the end of this section shows something in the wrong group.

In [ ]:
ACCIDENT_WORDS = ["collision", "accident", "damage", "spun", "contact", "debris"]
MECH_WORDS = ["gearbox", "brake", "engine", "power unit", "suspension", "electrical", "wheel", "driveshaft",
              "turbo", "hydraulic", "oil", "wing", "mechanical", "power loss", "puncture", "radiator",
              "transmission", "exhaust", "water", "fuel", "clutch", "differential", "overheat", "cooling",
              "vibration", "steering", "drivetrain", "energy", "battery", "technical", "tyre", "tire",
              "floor", "undertray", "plank", "fire", "leak", "pressure"]
DNS_WORDS = ["did not start", "withdrew", "illness", "did not qualify", "excluded"]

def status_group(s):
    if pd.isna(s):
        return "Unknown"
    t = str(s).strip().lower()
    if t == "finished":
        return "Finished"
    if t == "lapped" or re.match(r"^\+\d+ laps?$", t):
        return "Lapped"
    if "disqualif" in t:
        return "Disqualified"
    if t in DNS_WORDS:
        return "Withdrawn/DNS"
    if any(k in t for k in ACCIDENT_WORDS):
        return "Accident/Collision"
    if t == "retired":
        return "Retired (unspecified)"
    if any(k in t for k in MECH_WORDS):
        return "Mechanical"
    return "Other"

df["status_group"] = df["status_raw"].map(status_group)
(df.groupby(["status_group", "status_raw"]).size().rename("rows").reset_index()
   .sort_values(["status_group", "rows"], ascending=[True, False]))

## 9. Retirement flags and laps completed
- `finished_race`: Finished, or finished a lap or more down.
- `did_not_start`: withdrawn (`W`) or a DNS-type status.
- `retired`: did not finish, was not disqualified, and did start. A retired car can still be *classified* (for example, a driver who stopped on lap 52 of 56), so `retired` is based on `Status`, not on `classified_position`.

In [ ]:
df["laps_completed"] = pd.to_numeric(df["laps_completed"], errors="coerce")
df["race_laps"] = df.groupby(KEY3)["laps_completed"].transform("max")
df["laps_completed_share"] = (df["laps_completed"] / df["race_laps"]).round(3)

df["finished_race"] = df["status_group"].isin(["Finished", "Lapped"])
df["disqualified"] = df["status_group"].eq("Disqualified")
df["did_not_start"] = df["classification_code"].eq("W") | df["status_group"].eq("Withdrawn/DNS")
df["retired"] = ~df["finished_race"] & ~df["disqualified"] & ~df["did_not_start"]

# laps behind the winner, for cars that finished
df["laps_down"] = np.where(df["finished_race"], df["race_laps"] - df["laps_completed"], np.nan)

df[["finished_race", "retired", "did_not_start", "disqualified"]].sum().to_frame("drivers")

## 10. Time columns
In FastF1 results, `Time` holds the **winner's total race time**, and for the other lead-lap finishers the **gap to the winner**. Lapped and retired cars have no time.

In [ ]:
df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
winner = df["finish_position"].eq(1)

df["race_time_s"] = df["time_s"].where(winner)
df["gap_to_winner_s"] = df["time_s"].where(~winner)
df.loc[winner, "gap_to_winner_s"] = 0.0

print("Winners without a race time:", int((winner & df["race_time_s"].isna()).sum()))

## 11. Points, reduced-points races and outcome flags
A normal Grand Prix awards 101 or 102 points in total (25+18+15+12+10+8+6+4+2+1, plus 1 for the fastest lap when awarded; the fastest-lap point was dropped from 2025). A race with far fewer points was stopped early (Belgium 2021 scored half points) and should be treated with care in the models.

In [ ]:
df["points"] = pd.to_numeric(df["points"], errors="coerce")
df["race_points_total"] = df.groupby(KEY3)["points"].transform("sum")
df["reduced_points_race"] = df["session"].eq("Race") & (df["race_points_total"] < 90)

df["positions_gained"] = df["grid_position"] - df["finish_position"]   # positive = gained places
df["win"] = df["finish_position"].eq(1)
df["podium"] = df["finish_position"].le(3)
df["top10"] = df["finish_position"].le(10)
df["scored_points"] = df["points"].gt(0)

print("Total points per race (Race sessions):")
print(df[df["session"].eq("Race")].drop_duplicates(KEY3)["race_points_total"].value_counts().sort_index().to_string())
print("\nReduced-points races:")
df[df["reduced_points_race"]].drop_duplicates(KEY3)[["year", "round", "event", "race_points_total", "race_laps"]]

## 12. Quality checks

In [ ]:
problems = []

n = df.groupby(KEY3).size()
print("Drivers per race:", n.value_counts().to_dict())
if (n < 18).any():
    problems.append("some races have fewer than 18 drivers")
odd = n[n != 20].rename("drivers").reset_index().merge(
    df.drop_duplicates(KEY3)[KEY3 + ["event"]], on=KEY3)
if len(odd):
    print("\nRaces that do not have 20 drivers (check these are real, e.g. a car that did not take part):")
    print(odd.to_string(index=False))

dup_pos = df.dropna(subset=["finish_position"]).duplicated(KEY3 + ["finish_position"]).sum()
print("Duplicate finishing positions within a race:", dup_pos)
if dup_pos:
    problems.append("duplicate finishing positions")

winners = df.groupby(KEY3)["win"].sum()
print("Races without exactly one winner:", int((winners != 1).sum()))
if (winners != 1).any():
    problems.append("races without exactly one winner")

print("\nMissing values in the columns the team uses:")
check = ["driver", "team", "grid_position", "finish_position", "status_raw", "points", "laps_completed"]
print(df[check].isna().sum().to_string())
print("\nProblems found:", problems if problems else "none")

## 13. Cross-check with the laps table (optional)
Compares the drivers and the number of laps in `results` with the laps table, so Dorcas and you can see where the two disagree. Runs only if `laps_<year>.csv` exists.

In [ ]:
xs = []
for y in YEARS:
    p = COMBINED_DIR / f"laps_{y}.csv"
    if not p.exists():
        continue
    lp = pd.read_csv(p, usecols=["year", "round", "session", "Driver", "LapNumber"])
    lp = (lp.groupby(KEY3 + ["Driver"])["LapNumber"].max().rename("max_lap_in_laps_table")
            .reset_index().rename(columns={"Driver": "driver"}))
    xs.append(lp)

if xs:
    lps = pd.concat(xs, ignore_index=True)
    lps["driver"] = lps["driver"].astype("string").str.upper()
    df_x = df[df["year"].isin(lps["year"].unique())]          # only years that have a laps file
    both = df_x.merge(lps, on=KEY4, how="outer", indicator=True)
    both = both[both["session"].isin(df_x["session"].unique())]
    print("Drivers only in results:", int((both["_merge"] == "left_only").sum()))
    print("Drivers only in laps   :", int((both["_merge"] == "right_only").sum()))
    m = both[both["_merge"] == "both"].copy()
    m["lap_diff"] = m["laps_completed"] - m["max_lap_in_laps_table"]
    print("\nResults laps minus laps-table laps (0 = they agree):")
    print(m["lap_diff"].value_counts().sort_index().to_string())
else:
    print("No laps_<year>.csv files found, skipping.")

## 14. Save the cleaned table

In [ ]:
FINAL = [
    "year", "round", "session", "event", "country", "location", "event_date",
    "driver", "driver_id", "driver_number", "full_name", "team", "team_id", "team_group",
    "grid_position_raw", "pit_lane_start", "grid_position", "grid_missing",
    "finish_position", "classified_position", "classification_code",
    "status_raw", "status_group", "finished_race", "retired", "did_not_start", "disqualified",
    "laps_completed", "race_laps", "laps_completed_share", "laps_down",
    "points", "scored_points", "win", "podium", "top10", "positions_gained",
    "race_time_s", "gap_to_winner_s", "race_points_total", "reduced_points_race",
]
out = df[FINAL].sort_values(KEY4).reset_index(drop=True)
out.to_csv(OUT_DIR / "results_clean.csv", index=False)
print("Saved", OUT_DIR / "results_clean.csv", out.shape)
out.head()

## 15. Summary: why drivers retired
**Note:** from 2023 the source mostly reports `Retired` without a cause, so the cause split is only reliable for 2021 and 2022. For the retirement model use the single `retired` flag (and `laps_completed`), not the cause.

In [ ]:
summary = (out[out["retired"]].groupby(["year", "status_group"]).size().unstack(fill_value=0))
summary["total_retirements"] = summary.sum(axis=1)
summary

In [ ]:
summary.drop(columns="total_retirements").plot(kind="bar", stacked=True, figsize=(8, 4),
                                               title="Retirements by cause and year")